# Ollama GPU Sunucusu — Colab Runner

Bu defterin tek işi: [pqa](https://github.com/cxrbon16/pqa) pipeline'ının kullandığı QA üretici
modeli Ollama ile ayağa kaldırıp **Cloudflare Quick Tunnel** ile dışarıya (bu Colab dışındaki bir
makineye) açmak. Pipeline'ın kendisi (`fetch/passages/generate/filter/solve/band/publish`) artık
**burada değil**, tünelin ucundaki dış ortamda çalışıyor — bu defter sadece model + GPU sağlıyor.

**Önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir` menüsünden **G4** (NVIDIA RTX PRO 6000,
~96GB VRAM) seç. Sonra hücreleri sırayla çalıştır, en sonda çıkan `https://xxxx.trycloudflare.com`
URL'sini dış ortamdaki pipeline'ın `config.yaml` → `base_url` alanına ver.

In [ ]:
!nvidia-smi

## 1. Ollama'yı kur

Kullanılacak model: `gemma4:31b-it-bf16` (Gemma 4'ün en güçlü dense boyutu, 16-bit — G4'ün
~96GB VRAM'ı bunu rahat kaldırır). Tag [ollama.com/library](https://ollama.com/library) üzerinden
doğrulandı; Gemma 4'ün `-it-` (instruct) varyantları quantization sonekini `31b-it-bf16` gibi
ortada taşıyor, salt `31b-bf16` **geçersizdir**.

**İndirme ~63GB** — bant genişliğine göre 15-30 dakika sürebilir.

In [ ]:
!df -h /

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd lshw pciutils  # zstd: ollama install.sh çıkarma için; lshw/pciutils: GPU tespit uyarısını gidermek için
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || echo "UYARI: ollama PATH'te bulunamadı, yukarıdaki kurulum çıktısını kontrol et"

## 2. Sunucuyu başlat

In [ ]:
import os, shutil, subprocess, time, urllib.request

os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

ollama_bin = shutil.which("ollama") or next(
    (p for p in ("/usr/local/bin/ollama", "/usr/bin/ollama") if os.path.exists(p)), None
)
if ollama_bin is None:
    raise RuntimeError(
        "ollama binary bulunamadı. Bir önceki hücredeki kurulumu tekrar çalıştır "
        "(curl -fsSL https://ollama.com/install.sh | sh) ve çıktısında hata olup olmadığına bak."
    )

log_path = "/content/ollama_serve.log"
log_file = open(log_path, "w")
ollama_proc = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    # Colab'da bir hücreyi durdurmak (SIGINT) aynı process group'taki arka plan sürecini de
    # öldürebilir — start_new_session bunu ayrı bir session'a alıp bu sorundan korur. Sunucunun
    # "could not connect to ollama server" hatasıyla sessizce ölmesinin en sık nedeni budur.
    start_new_session=True,
)

for _ in range(30):
    if ollama_proc.poll() is not None:
        raise RuntimeError(
            f"ollama serve başlatılamadı (çıkış kodu {ollama_proc.returncode}). Log:\n"
            + open(log_path).read()
        )
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve 30 saniyede ayağa kalkmadı. Log:\n" + open(log_path).read())

print("ollama serve pid:", ollama_proc.pid, "| binary:", ollama_bin, "| log:", log_path)

# Ollama'nın GPU'yu gerçekten bulup bulmadığını logdan doğrula (kurulumdaki lspci/lshw
# uyarısı kozmetik olabilir — asıl kanıt bu satırlar).
print("\n--- ollama serve log (GPU ile ilgili satırlar) ---")
with open(log_path) as f:
    for line in f:
        if any(k in line.lower() for k in ("gpu", "cuda", "nvidia", "vram")):
            print(line.strip())

## 3. Modeli çek ve GPU'da çalıştığını doğrula

In [ ]:
!ollama pull gemma4:31b-it-bf16

In [ ]:
import json, subprocess

# Kesin GPU kontrolü: modeli bir kez çalıştır, sonra /api/ps'teki size_vram'e bak.
# size_vram > 0 ve size_vram ~= size ise model (neredeyse) tamamen GPU'da.
subprocess.run(["ollama", "run", "gemma4:31b-it-bf16", "merhaba"], capture_output=True, text=True, timeout=300)
ps = json.loads(subprocess.run(["curl", "-s", "http://localhost:11434/api/ps"], capture_output=True, text=True).stdout)
for m in ps.get("models", []):
    size_gb, vram_gb = m["size"] / 1e9, m["size_vram"] / 1e9
    print(f"{m['name']}: size={size_gb:.1f}GB, size_vram={vram_gb:.1f}GB -> "
          f"{'GPU üzerinde ✓' if vram_gb / size_gb > 0.9 else 'KISMEN/TAMAMEN CPU! sorunu araştır'}")

**Daha küçük GPU (T4/L4) kullanıyorsan:** `gemma4:31b-it-bf16` (~63GB) T4/L4'te (16-24GB) sığmaz.
Yukarıdaki iki hücrede model adını değiştir, örn. `gemma4:e4b-it-q8_0` (~10GB, T4'te rahat çalışır).

## 4. Tüneli aç

`localhost:11434`'ü (Ollama) genel bir URL'ye proxy'ler — hesap/kayıt gerektirmez. Bu hücreyi
**model çekimi ve GPU doğrulaması bittikten sonra** çalıştır; sunucu ayakta değilken tünel açılırsa
dış taraf sürekli `502 Bad Gateway` alır.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, re

tunnel_log = "/content/cloudflared.log"
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,  # hücre durdurulsa bile tünel ayakta kalsın
)

url = None
for _ in range(20):
    time.sleep(1)
    log = open(tunnel_log).read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if m:
        url = m.group(0)
        break

if url:
    print("Tünel URL'si (dış pipeline'ın config.yaml -> base_url alanına bunu ver):", url)
    print("örnek: " + url + "/v1")
else:
    print("URL 20 saniyede çıkmadı, log:\n" + open(tunnel_log).read())

## 5. Canlı tut

Bu sekmeyi kapatma / Colab'ı boşta bırakma — hem `ollama serve` hem `cloudflared` bu çalışma
zamanı sonlanınca ölür. Bağlantı kopar/URL değişirse bu hücreyi tekrar çalıştırıp yeni URL'yi
dış pipeline'a ilet:
```bash
!curl -s http://localhost:11434/api/tags | python3 -m json.tool   # sunucu hâlâ ayakta mı
!pgrep -af cloudflared                                             # tünel hâlâ ayakta mı
```